# MNIST MLP — RecPulse

Training a simple Multi-Layer Perceptron on MNIST from scratch using the RecPulse framework.

**Architecture:** 784 → 256 → 128 → 10 with ReLU, Dropout, and CrossEntropy loss.

**Expected result:** ~98% test accuracy in 10 epochs.

## 1. Imports

In [1]:
import sys
import time
sys.path.insert(0, '..')

import recpulse_cuda as rp
from recpulse.module import Module, Linear, Dropout
from recpulse.optim import Adam
from recpulse.scheduler import StepLR
from recpulse.data import load_mnist, get_batch

rp.manual_seed(42)

DEVICE = 'cuda'

print(f"RecPulse loaded (device={DEVICE})")

RecPulse loaded (device=cuda)


## 2. Load Data

MNIST: 60K training images, 10K test images. Each image is 28x28 pixels, flattened to 784. Pixel values normalized to [0, 1].

In [2]:
train_images, train_labels, test_images, test_labels = load_mnist('../data/mnist')

print(f"Train: {train_images.shape[0]} images, {train_images.shape[1]} features")
print(f"Test:  {test_images.shape[0]} images")
print(f"Labels: {sorted(set(train_labels[:100]))}")

Train: 60000 images, 784 features
Test:  10000 images
Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 3. Define Model

Three fully-connected layers with ReLU activations and Dropout for regularization.

In [3]:
class MLP(Module):
    def __init__(self):
        super().__init__()
        self.fc1 = Linear(784, 256)
        self.drop1 = Dropout(0.2)
        self.fc2 = Linear(256, 128)
        self.drop2 = Dropout(0.2)
        self.fc3 = Linear(128, 10)

    def forward(self, x):
        h = self.keep(self.fc1(x))
        h = self.keep(h.op_relu())
        h = self.keep(self.drop1(h))
        h = self.keep(self.fc2(h))
        h = self.keep(h.op_relu())
        h = self.keep(self.drop2(h))
        return self.fc3(h)

model = MLP()
num_params = sum(t.size for t in model.parameters())
print(f"Model parameters: {num_params:,}")
print(f"Tracked tensors: {list(model.tracked.keys())}")

Model parameters: 235,146
Tracked tensors: ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias']


## 4. Setup Training

Adam optimizer with learning rate decay every 5 epochs.

In [4]:
model.to(device=DEVICE)

optimizer = Adam(model.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

BATCH_SIZE = 64
NUM_EPOCHS = 10
num_train = train_images.shape[0]
num_batches = (num_train + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Device: {DEVICE}")
print(f"Optimizer: Adam (lr={optimizer.defaults['lr']})")
print(f"Scheduler: StepLR (step_size=5, gamma=0.5)")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per epoch: {num_batches}")

Device: cuda
Optimizer: Adam (lr=0.001)
Scheduler: StepLR (step_size=5, gamma=0.5)
Batch size: 64
Batches per epoch: 938


## 5. Helper: Compute Accuracy

In [5]:
def compute_accuracy(model, images, labels, batch_size=256):
    model.eval()
    correct = 0
    total = 0

    for b in range((len(labels) + batch_size - 1) // batch_size):
        batch_img, batch_lbl = get_batch(images, labels, b, batch_size)
        if batch_img is None:
            break

        if DEVICE != 'cpu':
            batch_img = batch_img.to(device=DEVICE)

        out = model(batch_img)

        if DEVICE != 'cpu':
            out = out.to(device='cpu')

        preds = out.to_numpy().argmax(axis=1)

        for i in range(len(preds)):
            if preds[i] == batch_lbl[i]:
                correct += 1
            total += 1

    model.train()
    return correct / total

## 6. Train

Full training loop: forward pass, cross-entropy loss, backward, gradient clipping, optimizer step. Test accuracy evaluated every 2 epochs.

In [6]:
history = {'loss': [], 'acc': [], 'lr': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_loss = 0
    start = time.time()

    for b in range(num_batches):
        batch_img, batch_lbl = get_batch(train_images, train_labels, b, BATCH_SIZE)
        if batch_img is None:
            break

        if DEVICE != 'cpu':
            batch_img = batch_img.to(device=DEVICE)

        model.zero_grad()
        out = model(batch_img)
        loss = out.op_cross_entropy_loss(batch_lbl)
        loss.backward()
        rp.clip_grad_norm(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.sum_all()
        num_loss += 1

    scheduler.step()
    elapsed = time.time() - start
    avg_loss = epoch_loss / num_loss
    history['loss'].append(avg_loss)
    history['lr'].append(scheduler.get_lr())

    acc = compute_accuracy(model, test_images, test_labels, batch_size=500)
    history['acc'].append((epoch, acc))
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}  loss={avg_loss:.4f}  acc={acc:.4f}  lr={scheduler.get_lr():.6f}  ({elapsed:.1f}s)")


Epoch  1/10  loss=0.3061  acc=0.9614  lr=0.001000  (2.5s)
Epoch  2/10  loss=0.1284  acc=0.9708  lr=0.001000  (2.5s)
Epoch  3/10  loss=0.0920  acc=0.9727  lr=0.001000  (2.5s)
Epoch  4/10  loss=0.0735  acc=0.9750  lr=0.001000  (2.5s)
Epoch  5/10  loss=0.0640  acc=0.9783  lr=0.000500  (2.5s)
Epoch  6/10  loss=0.0409  acc=0.9810  lr=0.000500  (2.5s)
Epoch  7/10  loss=0.0354  acc=0.9802  lr=0.000500  (2.5s)
Epoch  8/10  loss=0.0333  acc=0.9817  lr=0.000500  (2.5s)
Epoch  9/10  loss=0.0288  acc=0.9806  lr=0.000500  (2.6s)
Epoch 10/10  loss=0.0270  acc=0.9813  lr=0.000250  (2.6s)


## 7. Results

In [7]:
final_acc = compute_accuracy(model, test_images, test_labels, batch_size=500)
print(f"Final test accuracy: {final_acc:.2%}")
print(f"Final training loss: {history['loss'][-1]:.4f}")
print()
print("Loss per epoch:")
for i, l in enumerate(history['loss']):
    print(f"  {i+1:2d}: {l:.4f}")
print()
print("Test accuracy checkpoints:")
for epoch, acc in history['acc']:
    print(f"  Epoch {epoch+1}: {acc:.2%}")

Final test accuracy: 98.13%
Final training loss: 0.0270

Loss per epoch:
   1: 0.3061
   2: 0.1284
   3: 0.0920
   4: 0.0735
   5: 0.0640
   6: 0.0409
   7: 0.0354
   8: 0.0333
   9: 0.0288
  10: 0.0270

Test accuracy checkpoints:
  Epoch 1: 96.14%
  Epoch 2: 97.08%
  Epoch 3: 97.27%
  Epoch 4: 97.50%
  Epoch 5: 97.83%
  Epoch 6: 98.10%
  Epoch 7: 98.02%
  Epoch 8: 98.17%
  Epoch 9: 98.06%
  Epoch 10: 98.13%


## 8. Save Model

In [8]:
rp.save(model.tracked, "../data/mnist_mlp.rpt")
print("Model saved to data/mnist_mlp.rpt")

# Verify reload
from recpulse.serialize import save, load
model2 = MLP()
model2.load_state(rp.load("../data/mnist_mlp.rpt"))
model2.to(device=DEVICE)
reload_acc = compute_accuracy(model2, test_images, test_labels, batch_size=500)
print(f"Reloaded model accuracy: {reload_acc:.2%}")

Model saved to data/mnist_mlp.rpt
Reloaded model accuracy: 98.13%
